# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadFaizan0023/FlyRank_ML_internship_repo/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: 4 - Logistic Regression
Reason: Handling Multi-class classification task lane as my final output - 'decline_Score' range from (0-5)

**1.1: Import libraries**

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix
import numpy as np
import pandas as pd
import os, getpass
import duckdb

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Grouped by client: Because if i will take time split, there would be biasness in the output for specific clients. May perform good for some and may not for some. And maybe it memorizes the patterns for certain clients and may not perform well on test set later on.

**2.1: Read data**

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
data = pd.read_csv("data_for_baseline_action_score.csv")

In [3]:
data.shape

(2929582, 47)

In [5]:
display(data.head(10))

,client_hash_id,content_hash_id,gsc_impressions_prev30d,gsc_clicks_prev30d,gsc_avg_position_prev30d,ga4_pageviews_prev30d,ga4_sessions_prev30d,ga4_users_prev30d,'ga4_engaged_session_prev30,ga4_total_engagement_sec_prev30,...,sessions_paid_last30,sessions_ai_last30,scroll_events_last30,cpc_last30,backlinks_last30,ctr_last30,engagement_rate_last30,scroll_events_rate_last30,days_since_update_last30,decline_score
0,client_e5c2aa26a8598242,content_ffcbdf209cae5abc,214,3,3.962617,3,3,3,0,0,...,0,0,0,0.00,0.0,0.005181,0.0,0.0,18,17
1,client_e5c2aa26a8598242,content_ffcbdf209cae5abc,214,3,3.962617,3,3,3,0,0,...,0,0,0,0.00,0.0,0.005208,0.0,0.0,18,17
2,client_e5c2aa26a8598242,content_ffcbdf209cae5abc,214,3,3.962617,3,3,3,0,0,...,0,0,0,0.00,0.0,0.000000,0.0,0.0,18,17
3,client_e5c2aa26a8598242,content_9f49580281520710,83,3,1.951807,4,4,4,0,0,...,0,0,0,0.00,0.0,0.017857,0.0,0.0,14,17
4,client_e5c2aa26a8598242,content_9f49580281520710,83,3,1.951807,4,4,4,0,0,...,0,0,0,0.00,0.0,0.000000,0.0,0.0,14,17
5,client_e5c2aa26a8598242,content_ffcbdf209cae5abc,214,3,3.962617,3,3,3,0,0,...,0,0,0,0.00,0.0,0.000000,0.0,0.0,18,17
6,client_e5c2aa26a8598242,content_5e0b5a0c9de5ce66,141,1,3.851064,1,1,1,0,0,...,0,0,0,0.05,0.0,0.000000,0.0,0.0,14,17
7,client_e5c2aa26a8598242,content_29a2823ec2077049,522,3,0.440613,3,3,3,0,0,...,0,0,0,1.34,0.0,0.000000,0.0,0.0,6,17
8,client_e5c2aa26a8598242,content_29a2823ec2077049,522,3,0.440613,3,3,3,0,0,...,0,0,0,1.34,0.0,0.005587,0.0,0.0,6,17
9,client_e5c2aa26a8598242,content_29a2823ec2077049,522,3,0.440613,3,3,3,0,0,...,0,0,0,1.34,0.0,0.004049,0.0,0.0,6,17


**2.2: Handle missing values**

In [6]:
data.columns

Index(['client_hash_id', 'content_hash_id', 'gsc_impressions_prev30d',
       'gsc_clicks_prev30d', 'gsc_avg_position_prev30d',
       'ga4_pageviews_prev30d', 'ga4_sessions_prev30d', 'ga4_users_prev30d',
       ''ga4_engaged_session_prev30', 'ga4_total_engagement_sec_prev30',
       'sessions_organic_prev30', 'sessions_direct_prev30',
       'sessions_referral_prev30', 'sessions_social_prev30',
       'sessions_paid_prev30', 'sessions_ai_prev30', 'scroll_events_prev30',
       'content_created_date', 'content_updated_date', 'cpc_prev30',
       'backlinks_prev30', 'ctr_prev30', 'engagement_rate_prev30',
       'scroll_events_rate_prev30', 'days_since_update_prev30',
       'gsc_impressions_last30d', 'gsc_clicks_last30d',
       'gsc_avg_position_last30d', 'ga4_pageviews_last30d',
       'ga4_sessions_last30d', 'ga4_users_last30d',
       ''ga4_engaged_session_last30', 'ga4_total_engagement_sec_last30',
       'sessions_organic_last30', 'sessions_direct_last30',
       'sessions_referr

In [24]:
data["cpc_prev30"].isnull().sum()

np.int64(39006)

In [25]:
data['backlinks_prev30'].isnull().sum()

np.int64(839490)

In [27]:
data['engagement_rate_prev30'].isnull().sum()

np.int64(29333)

In [28]:
data['scroll_events_rate_prev30'].isnull().sum()

np.int64(3501)

In [45]:
data["cpc_last30"].isnull().sum()

np.int64(39006)

In [46]:
data["backlinks_last30"].isnull().sum()

np.int64(839490)

In [48]:
data["engagement_rate_last30"].isnull().sum()

np.int64(32574)

In [49]:
data["scroll_events_rate_last30"].isnull().sum()

np.int64(1691)

In [3]:
data = data.dropna(subset=['cpc_prev30']).reset_index(drop=True)

In [4]:
data = data.dropna(subset=['backlinks_prev30']).reset_index(drop=True)

In [5]:
data = data.dropna(subset=['engagement_rate_prev30']).reset_index(drop=True)

In [6]:
data = data.dropna(subset=['scroll_events_rate_prev30']).reset_index(drop=True)

In [7]:
data = data.dropna(subset=['engagement_rate_last30']).reset_index(drop=True)

In [8]:
data = data.dropna(subset=['scroll_events_rate_last30']).reset_index(drop=True)

check again

In [56]:
data["cpc_prev30"].isnull().sum()

np.int64(0)

In [57]:
data['backlinks_prev30'].isnull().sum()

np.int64(0)

In [58]:
data['engagement_rate_prev30'].isnull().sum()

np.int64(0)

In [59]:
data['scroll_events_rate_prev30'].isnull().sum()

np.int64(0)

In [60]:
data["cpc_last30"].isnull().sum()

np.int64(0)

In [61]:
data["backlinks_last30"].isnull().sum()

np.int64(0)

In [69]:
data["engagement_rate_last30"].isnull().sum()

np.int64(0)

In [70]:
data["scroll_events_rate_last30"].isnull().sum()

np.int64(0)

**2.3: Train-Test split = > Client hold-out split**

In [9]:
# Set random seed for reproducibility
np.random.seed(42)

# Get unique clients
unique_clients = data['client_hash_id'].unique()

# Shuffle and split client IDs (80/20)
np.random.shuffle(unique_clients)
split_idx = int(len(unique_clients) * 0.8)
train_clients = unique_clients[:split_idx]
test_clients = unique_clients[split_idx:]

# Split the dataset based on client groups
train_data = data[data['client_hash_id'].isin(train_clients)].copy()
test_data = data[data['client_hash_id'].isin(test_clients)].copy()

print(f"Total unique clients: {len(unique_clients)}")
print(f"Training clients: {len(train_clients)} | Rows: {len(train_data)}")
print(f"Testing clients: {len(test_clients)} | Rows: {len(test_data)}")

# Verify that there is no client overlap
overlap = set(train_data['client_hash_id']).intersection(set(test_data['client_hash_id']))
print(f"Client overlap between train and test: {len(overlap)}")

Total unique clients: 27
Training clients: 21 | Rows: 3127466
Testing clients: 6 | Rows: 146461
Client overlap between train and test: 0


In [11]:
train_data.shape

(3127466, 47)

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**3.1: Train X-y split**

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
X_train = train_data.drop(columns=['client_hash_id', 'content_hash_id', 'content_created_date', 'content_updated_date', 'decline_score'], axis=1)
y_train = train_data['decline_score']

**3.3: Model: LogisticRegression**

**3.3.1: Model training**

In [13]:
logistic_regression_model = LogisticRegression(max_iter=1000, class_weight='balanced', multi_class='multinomial')
logistic_regression_model.fit(X_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


KeyboardInterrupt: 

**3.3.2: Model predictions and evaluation**

In [ ]:
y_pred = logistic_regression_model.predict(X_test)
y_proba = logistic_regression_model.predict_proba(X_test)


In [ ]:
f1 = f1_score(y_test, y_pred, average='macro')
auc = roc_auc_score(y_test, y_proba, multi_class='ovo')

print(f"F1: {f1:.4f}")
print(f"ROC-AUC: {auc:.4f}")

F1: 0.4740
ROC-AUC: 0.8702


**3.3.3: Comparison table**

In [ ]:
print("Actual (y_test) vs Predicted (y_pred):")
comparison_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
display(comparison_df.head(20))
display(comparison_df.tail(20))
print(f"Total rows: {len(comparison_df)}\n")

Actual (y_test) vs Predicted (y_pred):


,Actual,Predicted
0,5,4
2,5,4
3,5,4
8,5,4
22,5,4
23,5,5
24,5,4
38,5,4
39,5,4
40,5,5


,Actual,Predicted
1452735,0,1
1452737,0,2
1452739,0,2
1452741,0,2
1452747,0,0
1452748,0,2
1452750,0,2
1452756,0,0
1452757,0,0
1452758,0,1


Total rows: 635596



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

**4.1: Confusion matrix**

In [ ]:
print(confusion_matrix(y_test, y_pred))

[[ 7307  1591  1409     0     0     0]
 [ 6434 10352  3269  1729  1196   245]
 [ 8666 16581 79917  5735  3017  1572]
 [  555 12992 58407 66673 32157 12602]
 [    2  2521 12391 48041 66789 44980]
 [    0     9     2  1574 36828 90053]]


**4.2: Classification report**

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.32      0.71      0.44     10307
           1       0.24      0.45      0.31     23225
           2       0.51      0.69      0.59    115488
           3       0.54      0.36      0.43    183386
           4       0.48      0.38      0.42    174724
           5       0.60      0.70      0.65    128466

    accuracy                           0.51    635596
   macro avg       0.45      0.55      0.47    635596
weighted avg       0.52      0.51      0.50    635596



**Interpretation**

The classifier shows an asymmetric precision-recall pattern that varies by class position, not a uniform extremes-vs-middle split. Class 0 has high recall (71%) but low precision (32%): the model over-predicts this class, catching most true 0s but misfiring on cases that belong to classes 1 and 2. Class 5, by contrast, now performs strongly on both fronts, recall 70% and precision 60%, making it the best-separated class in the matrix, with little bleed from neighboring classes. Classes 3 and 4 show a mixed pattern: moderate precision (54%, 48%) but weak recall (36%, 38%), meaning predictions for these classes are reasonably trustworthy, but most true cases get pulled away — class 3's true instances split heavily into classes 2 (58,407) and 4 (32,157), and class 4's into classes 3 (48,041) and 5 (44,980). Class 1 is the weakest class overall, with both low precision (24%) and low recall (45%): it absorbs heavy misclassification from classes 0 and 2, and its own true cases scatter across 0, 2, and 3 rather than concentrating.

Support remains heavily skewed toward classes 2-4 (473,598 of 635,596 rows), so the 0.51 accuracy and 0.50 weighted F1 are dominated by mid-range performance, while the 0.47 macro F1 (matching the reported 0.4740) shows that once class size is ignored, class 1's weakness and class 0's imprecision drag the average down noticeably.

Combined with the confusion matrix, this indicates the model still treats decline severity as an ordinal continuum it can rank (consistent with the 0.87 ROC-AUC), with most errors landing on adjacent classes but the boundary at class 5 is now well-resolved, while class 1 (and to a lesser extent class 0) remains the hardest region to separate from its neighbors.

## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.